## Indicators in jarjarquant

This notebook goes over how indicators are implemented in jarjarquant, how they can be used, and the common indicator workflows

---

In [ ]:
# Always start with importing the Jarjarquant class and initializing an instance
from jarjarquant import Jarjarquant
import logging

# Set global log level
logging.basicConfig(
    level=logging.INFO,  # Can be INFO, WARNING, ERROR
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s"
)

jjq = Jarjarquant()

---
The `list_indicator()` method provides an easy way to look at available indicators

In [ ]:
jjq.list_indicators()

---
In jarjarquant, each indicator is registered as an `IndicatorType`, which allows for type hints in the editor for a better developer experience. IndicatorType objects can also be used to get details about an indicator using utility method: `get_indicator_parameters()` 

In [ ]:
from jarjarquant import IndicatorType, get_indicator_parameters
import json

avwap_params = get_indicator_parameters(IndicatorType.ANCHORED_VWAP)
pi_params = get_indicator_parameters(IndicatorType.PRICE_INTENSITY)
stochrsi_params = get_indicator_parameters(IndicatorType.STOCHASTIC_RSI)
# print(json.dumps(avwap_params, indent=2, default=str))
print(json.dumps(pi_params, indent=2, default=str))
# print(json.dumps(stochrsi_params, indent=2, default=str))

In [ ]:
# Define a spec for the indicator - change any default values if needed
from jarjarquant import IndicatorSpec
avwap_2_21 = IndicatorSpec(IndicatorType.ANCHORED_VWAP, parameters={"threshold_value":0.02, "atr_period":21})
pi_3 = IndicatorSpec(IndicatorType.PRICE_INTENSITY, parameters={"smoothing_factor": 3})
stoch_rsi = IndicatorSpec(IndicatorType.STOCHASTIC_RSI)

---
### Indicator Design Evaluation Workflow

A common workflow in early stages of quantitative system development is to evaluate an indicator's design - as opposed to it's performance. A well designed indicator has good statistical properties (stationarity, normality, high entropy) across assets and regimes, which makes it suitable for machine learning algorithms and even rule based systems.

To evaluate an indicator across multiple samples use the `parallel_indicator_distribution_study()` method from `jjq.feature_evaluator`. The workflow looks slightly different based on the data source, but can broadly be broken down into 3 categories based on the type of data source: 1. External API, 2. Custom (Local) Data and 3. Synthetic Data 

#### 1. External API

#### 2. Custom (local) data

In [ ]:
from jarjarquant import BarSize, SampleRequest

# Import the SampleRequest class and specify asset_class, date_range, bar_sizes, and any asset_class specific filters
params = None
equity_sample = SampleRequest(sample_type="equities", start_date="2024-02-12", end_date="2024-02-14", bar_size=BarSize.ONE_MINUTE, n_samples = 10, params=params)

print(jjq.feature_evaluator.parallel_indicator_distribution_study(avwap_2_21, equity_sample))

In [ ]:
from jarjarquant.data_service import DataService
ds = DataService()
ds.load_from_database("ind_dist_studies_db")

---

### Single Indicator Evaluation Workflow

Once a set of primary indicator specs is selected, a comprehensive suite of performance evaluation tests can be run on a specified sample. 

To evaluate performance we need to generate signals. Signals in jarjarquant are straightforward since they rely on an underlying assumption: **the indicator design workflow ensures only stable, high entropy indicators continue to the evaluation phase.** 

In essence, the indicator is designed to capture information in the extremes. An indicator, for example, CMMA (close minus moving average) that signifies significant information when the values are extremely positive or negative is a well-designed indicator. This should be kept in mind when designing indicators.

The single indicator evaluation pipeline consists of the following:
- Define thresholds (either static or rolling) that will generate signals.
- Initial performance (spec + thresholds) on sample: Profit factor, Accuracy, Payoff ratio, Turnover (trades generated), Sharpe Ratio
- Optimized threshold performance with monte carlo permutation based p-values